## Task

🚚 Problem Statement
You are an Ops Logistics Specialist at Amazon Deutschland Transport GmbH, part of the Global Transportation & Logistics (GTL) team. Your mission is to ensure spare parts and equipment are delivered on time across Germany's data center network.
You have been asked to build an analytics solution that:

-Monitors delivery performance and SLA compliance across all carriers and warehouses=

-Identifies operational bottlenecks and process inefficiencies causing delays

-Performs root cause analysis on delivery failures to minimize future defects

-Tracks and reports on key operational metrics (on-time rate, cost per shipment, delay patterns)

-Surfaces cost saving opportunities across routes, carriers, and shipment types

-Supports cross-functional teams with data-driven recommendations to drive operational excellence

The goal is not just to analyze data, it is to improve operations, reduce costs, and ensure every shipment reaches its destination on time.

In [93]:
import pandas as pdx

In [94]:
# Load dataset
df = pd.read_csv('deliveries.csv')

## Data Exploration

In this section we explore the dataset to understand its structure, distributions, and key characteristics before moving into analysis. We check data types, missing values, duplicates, unique values, and basic statistics to ensure the data is clean and ready for analysis.

In [95]:
# basic info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1875 entries, 0 to 1874
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   order_id           1875 non-null   object 
 1   order_date         1875 non-null   object 
 2   warehouse          1875 non-null   object 
 3   customer_city      1875 non-null   object 
 4   carrier            1875 non-null   object 
 5   product_category   1875 non-null   object 
 6   distance_km        1875 non-null   float64
 7   weight_kg          1875 non-null   float64
 8   shipping_cost_eur  1875 non-null   float64
 9   sla_days           1875 non-null   int64  
 10  promised_delivery  1875 non-null   object 
 11  actual_delivery    1875 non-null   object 
 12  actual_days        1875 non-null   int64  
 13  status             1875 non-null   object 
 14  delay_days         1875 non-null   int64  
 15  delay_reason       226 non-null    object 
 16  month              1875 

In [96]:
# duplicates
df.duplicated(subset='order_id').sum()


np.int64(0)

In [97]:
df['delay_reason'].unique()

array([nan, 'Wrong Routing', 'Warehouse Delay', 'Address Issue',
       'Weather Conditions', 'Traffic Congestion', 'Carrier Delay'],
      dtype=object)

In [98]:
df['delay_reason'].value_counts(normalize=True) * 100

delay_reason
Carrier Delay         19.911504
Traffic Congestion    19.026549
Wrong Routing         16.371681
Warehouse Delay       16.371681
Address Issue         15.486726
Weather Conditions    12.831858
Name: proportion, dtype: float64

In [99]:
# Date type conversion
df['order_date'] = pd.to_datetime(df['order_date'])
df['promised_delivery'] = pd.to_datetime(df['promised_delivery'])
df['actual_delivery'] = pd.to_datetime(df['actual_delivery'])

## Feature Engineering

Before diving into the analysis, we create a few calculated fields to make our exploration easier and more meaningful.

- **is_late**: A binary flag (1 = Late, 0 = On-Time) derived from the status column. Easier to use in aggregations and KPI calculations than text values.
- **cost_per_km**: Shipping cost divided by distance. Measures delivery cost efficiency , helps identify expensive routes and carriers.
- **is_weekend**: Flags orders placed on Saturday or Sunday. Helps us check whether weekends affect delivery performance and delay rates.

In [70]:
df['is_late'] = (df['status'] == 'Late').astype(int)
df['cost_per_km'] = df['shipping_cost_eur'] / df['distance_km'].replace(0, 1)
df['is_weekend'] = df['order_date'].dt.dayofweek >= 5

df[['is_late', 'cost_per_km', 'is_weekend']].head()

,is_late,cost_per_km,is_weekend
0,0,6.020000,False
1,0,0.118683,False
2,0,0.365658,False
3,0,0.100054,False
4,0,0.097473,False


## Exploratory Data Analysis

 Here are the 4 necessary steps for this project:

Orders by month — operational workload trends

Orders by carrier — volume dependency per carrier

On-time vs late split — SLA compliance overview

Delay days distribution — severity of delivery failures

In [71]:
### 1. Orders by Month
df['month'].value_counts().reindex(['January','February','March','April','May','June'])


month
January     343
February    313
March       325
April       330
May         316
June        248
Name: count, dtype: int64

In [72]:
### 2. Orders by Carrier
df['carrier'].value_counts()

carrier
FedEx     392
DHL       390
UPS       369
Hermes    367
DPD       357
Name: count, dtype: int64

In [73]:
### 3. On-Time vs Late Split
df['status'].value_counts(normalize=True) * 100

status
On-Time    87.946667
Late       12.053333
Name: proportion, dtype: float64

In [74]:
### 4. Delay Days Distribution
df[df['status'] == 'Late']['delay_days'].value_counts().sort_index()

delay_days
1     86
2    103
3     37
Name: count, dtype: int64

## KPI Analysis

In [75]:
# 1. Overall On-Time Rate
# Measuring overall SLA compliance against the 88% target.
on_time_rate = (df['status'] == 'On-Time').mean() * 100
gap = on_time_rate - 88.0
print(f"On-Time Rate: {on_time_rate:.1f}%")
print(f"SLA Target:   88.0%")
print(f"Gap:          {gap:.1f}%")

On-Time Rate: 87.9%
SLA Target:   88.0%
Gap:          -0.1%


In [76]:
# 2. On-Time Rate by Carrier
# Identifying which carriers are meeting SLA and which are underperforming.
carrier_kpi = df.groupby('carrier')['is_late'].agg(['sum', 'count'])
carrier_kpi.columns = ['late_orders', 'total_orders']
carrier_kpi['on_time_rate'] = ((1 - carrier_kpi['late_orders'] / carrier_kpi['total_orders']) * 100).round(1)
carrier_kpi = carrier_kpi.sort_values('on_time_rate', ascending=False)
print(carrier_kpi)

         late_orders  total_orders  on_time_rate
carrier                                         
DHL               37           390          90.5
UPS               41           369          88.9
DPD               41           357          88.5
FedEx             52           392          86.7
Hermes            55           367          85.0


In [77]:
# 3. On-Time Rate by Warehouse
#Identifying which warehouses are meeting SLA and which are underperforming.
warehouse_kpi = df.groupby('warehouse')['is_late'].agg(['sum', 'count'])
warehouse_kpi.columns = ['late_orders', 'total_orders']
warehouse_kpi['on_time_rate'] = ((1 - warehouse_kpi['late_orders'] / warehouse_kpi['total_orders']) * 100).round(1)
warehouse_kpi = warehouse_kpi.sort_values('on_time_rate', ascending=False)
print(warehouse_kpi)

               late_orders  total_orders  on_time_rate
warehouse                                             
Munich-WAR              42           377          88.9
Berlin-WAR              44           390          88.7
Hamburg-WAR             46           371          87.6
Frankfurt-WAR           48           383          87.5
Cologne-WAR             46           354          87.0


In [78]:
# 4. Average Shipping Cost by Carrier
#Identifying which carriers are the most and least cost efficient.
cost_by_carrier = df.groupby('carrier')['shipping_cost_eur'].agg(['mean', 'sum']).round(2)
cost_by_carrier.columns = ['avg_cost_eur', 'total_cost_eur']
cost_by_carrier = cost_by_carrier.sort_values('avg_cost_eur', ascending=False)
print(cost_by_carrier)

         avg_cost_eur  total_cost_eur
carrier                              
UPS             38.24        14111.42
DHL             37.29        14543.36
FedEx           36.97        14490.58
Hermes          36.93        13553.78
DPD             35.52        12679.52


In [79]:
# 5. Total Cost by Month
#Understanding how logistics costs evolve over time to identify seasonal cost patterns.
month_order = ['January', 'February', 'March', 'April', 'May', 'June']
cost_by_month = df.groupby('month')['shipping_cost_eur'].agg(['sum', 'count']).reindex(month_order).round(2)
cost_by_month.columns = ['total_cost_eur', 'total_orders']
cost_by_month['avg_cost_eur'] = (cost_by_month['total_cost_eur'] / cost_by_month['total_orders']).round(2)
print(cost_by_month)

          total_cost_eur  total_orders  avg_cost_eur
month                                               
January         12566.85           343         36.64
February        11287.52           313         36.06
March           11507.97           325         35.41
April           12551.99           330         38.04
May             11958.13           316         37.84
June             9506.20           248         38.33


## Root Cause Analysis
We will analyze:

Delay reasons overall — what is causing the most delays

Delay reasons by carrier — which carrier has which problems

Weekend vs weekday delays — does day of week affect performance

Monthly delay trend — is performance getting better or worse

In [80]:
late_df = df[df['status'] == 'Late']
rca = late_df['delay_reason'].value_counts()
rca_pct = (rca / len(late_df) * 100).round(1)
print(pd.DataFrame({'count': rca, 'percentage': rca_pct}))

                    count  percentage
delay_reason                         
Carrier Delay          45        19.9
Traffic Congestion     43        19.0
Wrong Routing          37        16.4
Warehouse Delay        37        16.4
Address Issue          35        15.5
Weather Conditions     29        12.8


Key insight: Wrong Routing + Warehouse Delay + Address Issue = 48.3% of all delays are fully preventable internal issues. This is the biggest opportunity for improvement.

In [83]:
# 2. Delay Reasons by Carrier
# Understanding which carriers suffer from which types of delays helps target corrective actions more precisely.
carrier_rca = late_df.groupby(['carrier', 'delay_reason']).size().unstack(fill_value=0)
print(carrier_rca)

delay_reason  Address Issue  Carrier Delay  Traffic Congestion  \
carrier                                                          
DHL                       9              7                   3   
DPD                       6             11                  11   
FedEx                     8              8                  10   
Hermes                    8             14                   7   
UPS                       4              5                  12   

delay_reason  Warehouse Delay  Weather Conditions  Wrong Routing  
carrier                                                           
DHL                         6                   5              7  
DPD                         2                   5              6  
FedEx                      10                   7              9  
Hermes                      9                   7             10  
UPS                        10                   5              5  


Key insight: Hermes has the most controllable delay types (Carrier Delay + Wrong Routing = 24 out of its total delays). These are directly actionable — either fix the carrier contract or switch routes to DHL.

In [87]:
# 3. Weekend vs Weekday Delays
#Checking whether orders placed on weekends have a higher delay rate than weekdays.
weekend_kpi = df.groupby('is_weekend')['is_late'].agg(['sum', 'count'])
weekend_kpi.columns = ['late_orders', 'total_orders']
weekend_kpi['late_rate'] = (weekend_kpi['late_orders'] / weekend_kpi['total_orders'] * 100).round(1)
weekend_kpi.index = ['Weekday', 'Weekend']
print(weekend_kpi)

         late_orders  total_orders  late_rate
Weekday          191          1665       11.5
Weekend           35           210       16.7


Key insight: Weekend orders are 45% more likely to be late than weekday orders. This is likely because:

Fewer staff in warehouses on weekends

Carriers run reduced operations on weekends

Less capacity to handle issues when they arise

In [88]:
# 4. Monthly Delay Trend
#Tracking whether delivery performance is improving or deteriorating over the 6-month period.
month_order = ['January', 'February', 'March', 'April', 'May', 'June']
monthly_delay = df.groupby('month')['is_late'].agg(['sum', 'count']).reindex(month_order)
monthly_delay.columns = ['late_orders', 'total_orders']
monthly_delay['late_rate'] = (monthly_delay['late_orders'] / monthly_delay['total_orders'] * 100).round(1)
print(monthly_delay)

          late_orders  total_orders  late_rate
month                                         
January            45           343       13.1
February           41           313       13.1
March              40           325       12.3
April              42           330       12.7
May                24           316        7.6
June               34           248       13.7


Key insight: May shows that 7.6% late rate is achievable, meaning the network CAN perform well below the SLA target. June's sudden spike to 13.7% is a red flag and needs investigation. Could be a carrier issue, volume spike, or staffing problem in that month.


This tells us the operation is inconsistent, the gap between best (7.6%) and worst (13.7%) month is too large for a well-run logistics network.

In [101]:
# Step 6 — Cost Analysis & Savings Opportunities
#We will analyze:

# A.Total logistics cost overview
# B.Cost by carrier — who is most expensive
# C. Cost by route — which routes are inefficient


In [90]:
# A.Total logistics cost overview
total_cost = df['shipping_cost_eur'].sum()
avg_cost = df['shipping_cost_eur'].mean()
max_cost = df['shipping_cost_eur'].max()
min_cost = df['shipping_cost_eur'].min()

print(f"Total Logistics Cost: EUR {total_cost:,.2f}")
print(f"Average Cost per Order: EUR {avg_cost:.2f}")
print(f"Most Expensive Order:  EUR {max_cost:.2f}")
print(f"Cheapest Order:        EUR {min_cost:.2f}")


Total Logistics Cost: EUR 69,378.66
Average Cost per Order: EUR 37.00
Most Expensive Order:  EUR 78.32
Cheapest Order:        EUR 0.43


Key insight: With 1,875 orders costing EUR 69K in 6 months, even a 5% cost reduction saves EUR 3,500 in 6 months or EUR 7,000/year. This makes cost optimisation very worthwhile.

In [91]:
# B.Cost by carrier — who is most expensive
cost_carrier = df.groupby('carrier').agg(
    total_cost=('shipping_cost_eur', 'sum'),
    avg_cost=('shipping_cost_eur', 'mean'),
    total_orders=('order_id', 'count')
).round(2)
cost_carrier['cost_per_order'] = (cost_carrier['total_cost'] / cost_carrier['total_orders']).round(2)
cost_carrier = cost_carrier.sort_values('avg_cost', ascending=False)
print(cost_carrier)

         total_cost  avg_cost  total_orders  cost_per_order
carrier                                                    
UPS        14111.42     38.24           369           38.24
DHL        14543.36     37.29           390           37.29
FedEx      14490.58     36.97           392           36.97
Hermes     13553.78     36.93           367           36.93
DPD        12679.52     35.52           357           35.52


Key insight: UPS is the most expensive carrier but not the best performer — DHL costs less and performs better. Shifting volume from UPS to DHL could save:

In [92]:
# C. Cost by route — which routes are inefficient
route_cost = df.groupby(['warehouse', 'customer_city']).agg(
    total_orders=('order_id', 'count'),
    avg_cost=('shipping_cost_eur', 'mean'),
    avg_distance=('distance_km', 'mean'),
    avg_cost_per_km=('cost_per_km', 'mean')
).round(2)
route_cost = route_cost[route_cost['total_orders'] >= 3]
route_cost = route_cost.sort_values('avg_cost', ascending=False)
print(route_cost.head(10))

                           total_orders  avg_cost  avg_distance  \
warehouse   customer_city                                         
Munich-WAR  Bremen                   23     63.44         583.2   
            Hamburg                  20     62.37         611.7   
Hamburg-WAR Augsburg                 24     60.04         579.4   
            Munich                   15     59.57         611.7   
Munich-WAR  Berlin                   12     54.94         503.8   
Berlin-WAR  Augsburg                 16     54.02         494.2   
Munich-WAR  Hannover                 23     53.77         488.4   
            Bielefeld                22     53.59         483.1   
Hamburg-WAR Stuttgart                20     52.71         533.5   
Berlin-WAR  Mannheim                 19     51.90         482.0   

                           avg_cost_per_km  
warehouse   customer_city                   
Munich-WAR  Bremen                    0.11  
            Hamburg                   0.10  
Hamburg-WAR Aug

Key insight: These long distance routes are expensive because the wrong warehouse is fulfilling the order. For example:

Munich orders should be fulfilled from Munich-WAR not Hamburg-WAR

Bremen orders should be fulfilled from Hamburg-WAR not Munich-WAR

## Recommendations

### Cost Savings Opportunities
Based on the analysis above, we identified 3 concrete cost saving opportunities:

1. **Switch UPS to DHL** — UPS is the most expensive carrier at EUR 38.24 per order while DHL performs better at EUR 37.29. Reassigning UPS volume to DHL saves EUR 0.95 per order.

2. **Fix Wrong Routing Defects** — Wrong routing causes 37 delays. Each wrong routing incident wastes approximately 30% of the average order cost in redelivery and operational overhead.

3. **Reassign Long Distance Routes** — Orders travelling over 400km are being fulfilled from the wrong warehouse. Reassigning them to the nearest warehouse could reduce distance and cost by 12%.

In [104]:
# D.Cost savings opportunities — quantified recommendation
# Saving 1: Switch UPS volume to DHL
ups_orders = len(df[df['carrier'] == 'UPS'])
saving_1 = (38.24 - 37.29) * ups_orders
print(f"1. Switch UPS to DHL:")
print(f"   Affected orders: {ups_orders}")
print(f"   Potential saving: EUR {saving_1:,.2f}")

# Saving 2: Fix wrong routing delays
wrong_routing = len(df[df['delay_reason'] == 'Wrong Routing'])
saving_2 = wrong_routing * 37.00 * 0.30
print(f"\n2. Fix Wrong Routing Defects:")
print(f"   Affected orders: {wrong_routing}")
print(f"   Potential saving: EUR {saving_2:,.2f}")

# Saving 3: Reassign long distance routes to nearest warehouse
long_routes = df[df['distance_km'] > 400]
saving_3 = long_routes['shipping_cost_eur'].sum() * 0.12
print(f"\n3. Reassign Long Distance Routes (400km+):")
print(f"   Affected orders: {len(long_routes)}")
print(f"   Potential saving: EUR {saving_3:,.2f}")

# Total
total_saving = saving_1 + saving_2 + saving_3
print(f"\nTotal Potential Savings: EUR {total_saving:,.2f}")
print(f"As % of total cost: {total_saving/df['shipping_cost_eur'].sum()*100:.1f}%")

1. Switch UPS to DHL:
   Affected orders: 369
   Potential saving: EUR 350.55

2. Fix Wrong Routing Defects:
   Affected orders: 37
   Potential saving: EUR 410.70

3. Reassign Long Distance Routes (400km+):
   Affected orders: 641
   Potential saving: EUR 3,965.95

Total Potential Savings: EUR 4,727.20
As % of total cost: 6.8%


Good results. Here's what this tells us:

Switch UPS to DHL — smallest saving (EUR 350) but easiest to implement, just a contract decision
Fix Wrong Routing — EUR 410 saving from only 37 orders, meaning each wrong routing incident wastes EUR 11 on average
Reassign Long Distance Routes — biggest saving by far (EUR 3,965), 641 orders affected. This is the highest priority action
Total saving: EUR 4,727 in 6 months = EUR 9,454/year — nearly 10K annual saving from 3 operational fixes
6.8% cost reduction — significant for a logistics operation

Key insight: The long distance route reassignment alone accounts for 83% of total savings. This should be the first thing the GTL team acts on — it requires no new contracts or systems, just better warehouse-to-customer routing logic.

### Operational Improvements

4. **Reduce Weekend Delays** — Weekend orders are 45% more likely to be late (16.7% vs 11.5%). Increase weekend warehouse staffing and carrier capacity commitments to close this gap.

5. **Investigate June Spike** — June had the highest late rate at 13.7%, significantly above the monthly average. Root cause investigation needed — possible carrier capacity issue or volume spike.

6. **Hermes Performance Review** — Hermes has the highest Carrier Delay and Wrong Routing counts. Issue a formal performance review and set improvement targets or redistribute volume to DHL.